In [ ]:
def retrain_all():
    dataset_path = r"C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/Dataset_Face"

    embeddings = []
    names = []

    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        if not os.path.isdir(person_folder):
            continue

        for img_name in os.listdir(person_folder):
            img_path = os.path.join(person_folder, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue

            h, w = img.shape[:2]
            blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)
            net.setInput(blob)
            ln = net.getUnconnectedOutLayersNames()
            layer_outputs = net.forward(ln)

            boxes = []
            confidences = []

            for output in layer_outputs:
                for detection in output:
                    scores = detection[5:]
                    confidence = scores[0]
                    if confidence > 0.5:
                        box = detection[0:4] * np.array([w, h, w, h])
                        (centerX, centerY, width, height) = box.astype("int")
                        x = int(centerX - width / 2)
                        y = int(centerY - height / 2)
                        boxes.append([x, y, int(width), int(height)])
                        confidences.append(float(confidence))

            idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)
            if len(idxs) == 0:
                continue

            i = idxs.flatten()[0]
            (x, y) = (boxes[i][0], boxes[i][1])
            (w_box, h_box) = (boxes[i][2], boxes[i][3])
            face_crop = img[y:y+h_box, x:x+w_box]
            if face_crop.size == 0:
                continue

            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            face_resized = cv2.resize(face_rgb, (160,160))
            face_tensor = torch.tensor(face_resized).permute(2,0,1).float()/255.0
            face_tensor = (face_tensor - 0.5)/0.5
            face_tensor = face_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                embedding = resnet(face_tensor).cpu().numpy()[0]

            embeddings.append(embedding)
            names.append(person_name)

    if len(embeddings)==0:
        messagebox.showwarning("Peringatan","Tidak ada data embedding ditemukan.")
        return

    embeddings = np.array(embeddings)
    names = np.array(names)
    embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
    knn.fit(embeddings_norm, names)
    joblib.dump(knn, "face_classifier.pkl")

    messagebox.showinfo("Info","✅ Model berhasil dilatih ulang dan disimpan.")
